# Exploratorial Data Analysis

## Data Loading

In [ ]:
import matplotlib.pylab as plt
import missingno
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from mlt.config import RAW_DATA_DIR

In [ ]:
df = pd.read_csv(str(RAW_DATA_DIR) + "/telco_data.csv")

In [ ]:
df

## Data Cleaning

In [ ]:
# NUR DROPNA WÜRDE 12% DER DATEN ENFERNEN

# Dollar entfernen und in float konvertieren
df["MonthlyCharges"] = df["MonthlyCharges"].str.lstrip("$").astype(float)

# gender bezeichnungen einheitlich
df["gender"] = df["gender"].str.lower().str.strip()
df["gender"] = df["gender"].replace(
    {"m": "male", "male": "male", "f": "female", "female": "female"}
)

# Partner beizeichnungen einheitlich
df["Partner"] = df["Partner"].str.lower().str.strip()
df["Partner"] = df["Partner"].replace({"y": "yes", "n": "no"})

# Tenure month enfernen und in float konvertieren
df["tenure"] = df["tenure"].str.strip(" months").astype(float)

# Dollar hinten enfernen
df["TotalCharges"] = df["TotalCharges"].str.strip(" USD").astype(float)

# Senior citzien
df["SeniorCitizen"] = df["SeniorCitizen"].str.lower().str.strip()
df["SeniorCitizen"] = df["SeniorCitizen"].replace(
    {"y": "yes", "n": "no", "0": "no", "1": "yes"}
)

# Dependants yes/no
df["Dependents"] = df["Dependents"].str.lower().str.strip()
df["Dependents"] = df["Dependents"].replace({"y": "yes", "n": "no"})

# PhoneService yes/no
df["PhoneService"] = df["PhoneService"].str.lower().str.strip()
df["PhoneService"] = df["PhoneService"].replace({"y": "yes", "n": "no"})

In [ ]:
missingno.matrix(df)
# Bei Tenure und total charges aufüllen mit average
# Bei InternetService, Monthloy Charges, contract und Payment method aufüllen mit Modus

In [ ]:
imputer_mode = SimpleImputer(strategy="most_frequent")
df[["InternetService", "PaymentMethod", "Contract", "MonthlyCharges"]] = (
    imputer_mode.fit_transform(
        df[["InternetService", "PaymentMethod", "Contract", "MonthlyCharges"]]
    )
)

df.isnull().sum()
# Alle Daten aufgefüllt

In [ ]:
df = df.drop(["customerID", "customer_hash"], axis=1)
df

## Data Analysis and Correction

In [ ]:
plt.scatter(df["tenure"], df["TotalCharges"], s=1)
plt.title("Insgesamte Ausgaben der Kunden nach Abolänge")

# Seltsamte Punkte -> Rausschneiden
# Keine Wirkliche Informationen wenn man tenure schon hat, je länger man bleibt desto mehr zahlt man linear

In [ ]:
plt.scatter(df["tenure"], df["MonthlyCharges"], s=1)
plt.title("Monatliche Ausgaben der Kunden nach Abolänge")

# Seltsamte Punkte -> Rausschneiden
# Keine Wirkliche Informationen wenn man tenure schon hat, je länger man bleibt desto mehr zahlt man linear

In [ ]:
df.shape

In [ ]:
plt.hist(df["MonthlyCharges"], bins=100)
plt.title("Häufigkeit von Monatlichen Abrechnungen bei Kunden")
plt.xlabel("Monatliche Kosten in Dollar")
plt.ylabel("Häufigkeit")

# Ausresßer entfernen?

df = df[df["tenure"] < 90]
df = df[df["tenure"] > 0]

In [ ]:
df.shape

In [ ]:
target = df["Churn"]
target.hist()
plt.title("Anzahl der Churns/nicht Churns")

# Relativ ausgeglichene Daten

In [ ]:
pd.crosstab(df["gender"], df["Churn"], normalize="index")

# Kein Zusammenhang zwischen Gender und Churn

In [ ]:
pd.crosstab(df["Partner"], df["Churn"], normalize="index")

# Partner könnte wichtiges Feature sein, mit partner qird ca 13% mehr gekündigt

In [ ]:
df["Contract"].value_counts()

# Die Meisten sind auf Monatsabo

# Ordinal Encoding um reihenfolge zu behalten
df["Contract"] = (
    df["Contract"]
    .replace({"Month-to-month": 0, "One year": 1, "Two year": 2})
    .astype(float)
)

In [ ]:
test = pd.crosstab(df["Contract"], df["Churn"], normalize="index")
plt.plot(test.Yes)
plt.title("Vertragslänge vs Churn")
plt.ylabel("Warscheinlichkeit zum Churn")
plt.xlabel("Abonementvertragin Jahren")
test

In [ ]:
df = pd.get_dummies(df, columns=["PaymentMethod", "Churn"], dtype=int)
df

In [ ]:
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True)

# Electronic Check churnen mehr !

In [ ]:
sns.lineplot(data=df, x="tenure", y="Churn_Yes")
plt.xlabel("Tenure")
plt.ylabel("Churnwarscheinlichkeit")
plt.title("Mitgliedsdauer vs Churn")
plt.show()

In [ ]:
pd.crosstab(df["PaperlessBilling"], df["Churn_Yes"], normalize="index")

In [ ]:
for col in [
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]:
    print(pd.crosstab(df[col], df["Churn_Yes"], normalize="index"))
    print()


# Online Backup, Device Protection, TechSupport sind sehr korreliert
# Streaming nicht so wichtig

In [ ]:
for col in [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "SeniorCitizen",
]:
    print(col)
    print(pd.crosstab(df[col], df["Churn_Yes"], normalize="index"))
    print()

# Alte Menschen churnen mehr, Interservice, online security wichtig
# Menschen ohne internet churnen generell sehr wenig

In [ ]:
# Online Backup, Device Protection, TechSupport, Interservice, SeniorCitizen One hot encoden und die nicht gewollten droppen
df = pd.get_dummies(
    df,
    columns=[
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "InternetService",
        "SeniorCitizen",
        "Dependents",
        "OnlineSecurity",
        "PaperlessBilling",
    ],
    dtype=int,
)
df = df.drop(
    [
        "StreamingMovies",
        "StreamingTV",
        "MultipleLines",
        "PhoneService",
        "TotalCharges",
        "Partner",
        "gender",
    ],
    axis=1,
)
df.shape

In [ ]:
df = df.drop_duplicates()
df.shape